**Notebook Feito por:** MSc. Eng. Paulo de Souza Silva  
**Data:** Agosto de 2026  
**Conteúdo retirado e adaptado de livros e artigos sobre Galerkin Descontínuo**  
**Agradecimentos:** Um agradecimento ao Prof. Dr. Alberto Nogueira pela disponibilização dos scripts em Python para DG 1D

# **Slope Limiters**

## **Introdução**

Se olharmos pelo retrovisor da nossa jornada computacional, perceberemos que já construímos um motor numérico incrivelmente robusto para resolver Leis de Conservação. Nas aulas anteriores, nós consolidamos duas grandes frentes do Método de Galerkin Descontínuo (DG):

A Discretização Espacial: Criamos a nossa malha, mapeamos os elementos usando o Jacobiano e utilizamos os Polinômios de Legendre para construir as Matrizes de Massa e de Rigidez. Além disso, conectamos os elementos isolados através dos Fluxos Numéricos e das Matrizes de Elevação (Lift Matrices), culminando no nosso operador espacial $L_h(\mathbf{U}, t)$.

$$L_h = \frac{1}{J_e} \left[\int_{\Omega_{pd}} \phi_i \phi_j d \xi \right]^{-1} \left ( \int_{\Omega_{pd}} f \dfrac{\partial}{\partial \xi} \begin{Bmatrix} \phi_0 \\ \phi_1 \\ \vdots \\ \phi_P \end{Bmatrix} d \xi-  \tilde{f}_{e,e+1} \begin{Bmatrix} \phi^+_0 \\ \phi^+_1 \\ \vdots \\ \phi^+_P \end{Bmatrix} +  \tilde{f}_{e-1,e}\begin{Bmatrix} \phi^-_0 \\ \phi^-_1 \\ \vdots \\ \phi^-_P \end{Bmatrix} \right )$$

A Integração Temporal: Descobrimos que integradores clássicos de alta precisão, como o RK4, podem destruir propriedades de estabilidade não-linear. Para corrigir isso, implementamos a família de integradores Runge-Kutta SSP (como o SSPRK22 e o poderoso SSPRK104), que avançam no tempo garantindo que a variação total do sistema não cresça desenfreadamente.

Neste ponto, o nosso solver funciona perfeitamente para escoamentos suaves e problemas contínuos. No entanto, o que acontece quando a física do problema exige a formação de descontinuidades, como uma onda de choque supersônica ou o rompimento de uma barragem?

As equações hiperbólicas não-lineares, como as Equações de Euler ou a Equação de Burgers, possuem uma característica natural de "enrijecimento" das frentes de onda. Um perfil que começa como uma curva suave pode evoluir até se tornar um "degrau" vertical abrupto.

É aqui que a matemática dos polinômios entra em conflito com a física:

* O Dilema da Baixa Ordem: Se usarmos um polinômio de Grau 0 (uma reta constante por partes, equivalente ao Método de Volumes Finitos clássico de 1ª Ordem), a nossa simulação será extremamente estável e nunca irá oscilar. Contudo, esse esquema é tão dissipativo (possui tanta difusão numérica) que o degrau do choque é "borrado" e se transforma em uma rampa suave. Perdemos a capacidade de capturar a alta resolução do fenômeno.

* O Desastre da Alta Ordem: Quando utilizamos polinômios de Grau 1 ou superior para obtermos alta precisão, a matemática tenta interpolar uma curva polinomial suave sobre um degrau vertical. O resultado inevitável é o surgimento de oscilações espúrias severas logo antes e logo depois da descontinuidade. Esse efeito é classicamente conhecido como o Fenômeno de Gibbs.

> O Risco Computacional e Físico:  
> O Fenômeno de Gibbs não é apenas uma "feiura" gráfica. Em simulações de dinâmica dos fluidos, os "vales" dessas oscilações irreais (os chamados undershoots) podem fazer com que o solver calcule localmente uma densidade negativa, ou uma pressão abaixo do vácuo absoluto. Ao tentar calcular a velocidade do som termodinâmica no próximo passo do Runge-Kutta, o código extrairá a raiz quadrada de um número negativo e a simulação irá quebrar irreversivelmente (`NaN`).

Para resolver esse conflito, nós não precisamos abandonar a Alta Ordem, mas sim dotar o nosso código de um "sensor de segurança". Precisamos de uma rotina matemática que atue no espaço, analise as interfaces dos elementos e diga: "Atenção, há um choque aqui! Reduza a ordem polinomial apenas nesta célula para evitar oscilações, mas mantenha a alta precisão no resto do domínio".

Esses corretores espaciais inteligentes são conhecidos como **Limitadores de Inclinação** (*Slope Limiters*).

## **Slope Limiters**

> **Referência:** *Understand Slope Limiter - Graphically* (Ling Zou, 2021)

Se usarmos esquemas numéricos de alta ordem de precisão espacial (para obter resultados mais acurados em regiões onde o fluido se comporta de maneira suave), a matemática pura acaba gerando "oscilações espúrias" (overshoots e undershoots não físicos) perto dessas descontinuidades.

Para evitar isso, precisamos de um método Total Variance Diminishing (TVD), que, de forma simplificada, preserva a monotonicidade da solução e evita a criação de novos extremos locais que não existem na física do problema.

### **Algoritmo REA**

A equação do problema de advecção apresentada pelo formato de **volumes finitos** (ver **aula 08**) é:

$$U_i^{n+1} = U^n_i - \dfrac{\Delta t}{\Delta x} \Big (F_{i+\frac{1}{2}}-F_{i-\frac{1}{2}}  \Big)$$

e se usarmos o método Upwind com $a > 0$ temos:

$$U_i^{n+1} = U_i^n - a \frac{\Delta t}{\Delta x}(U_i^n - U_{i-1}^n) $$

a partir do qual $U_i^{n+1}$ pode ser atualizado. Esse processo pode ser ilustrado conforme mostrado na Figura.

> Adicionar figura 2 do artigo

Partimos das soluções conhecidas, $U_i^n$ (médias por célula), referentes ao passo de tempo $t_n$, e realizamos uma simples reconstrução constante por partes da solução em cada célula, ilustrada na Figura a. Em seguida, evoluímos a solução utilizando o método *upwind* ao longo do passo de tempo $\Delta t$ - isto é, todo o perfil da solução desloca-se para a direita (visto que $a > 0$) por uma distância $a\Delta t$, como mostra a Figura b. Por fim, calcula-se a média local em cada célula para obter a solução $U_i^{n+1}$ no instante $t_{n+1}$, conforme ilustrado na Figura c.


Esse processo é uma versão simplificada do algoritmo REA (Reconstruct-Evolve-Average):

* **Reconstrução:** Partimos dos valores médios conhecidos em cada célula ($U^n$) e reconstruímos uma função dentro dessa célula.  
* **Evolução:** Resolvemos a física exata ou aproximada por um passo de tempo $\Delta t$.  
* **Média:** Tiramos a média do resultado para o próximo passo de tempo.

Se a nossa reconstrução for apenas uma linha reta horizontal (constante por partes), o método é muito estável, mas tem precisão apenas de primeira ordem. Para obtermos segunda ordem, precisamos fazer uma **reconstrução linear por partes**.

### **Reconstrução Linear Por Partes e Regra TVD**

Para garantir a conservação da massa em cada célula, a função linear por partes reconstruída deve passar, por exemplo, pelo ponto $(x_i, U_i)$ na $i$-ésima célula, marcado como "x" na Figura 3a. 

> adicionar figura 3 do artigo

Assim, resta apenas um grau de liberdade que podemos ajustar: a inclinação da função linear por partes. Conforme ilustrado na Figura 3b, a reconstrução da função linear assemelha-se a uma situação em que o centro da barra está fixo, permitindo que ela gire no sentido horário ou anti-horário (indicado pelo símbolo de "mão") em torno desse ponto fixo.

Se a conservação de massa local for a única restrição, a inclinação pode assumir quaisquer valores. No entanto,
veremos que a inclinação não pode assumir valores arbitrários se quisermos construir um esquema TVD (de alta
resolução). Para esquemas TVD, a inclinação só pode ser escolhida dentro de uma faixa limitada;
é daí que vem o nome "limitador de inclinação" (*slope limiter*). 


Antes de discutirmos o limitador de inclinação, várias grandezas importantes devem ser apresentadas e discutidas. E para nos ajudar, tome como base a Figura

> Adicionar a Figura 4 do artigo

Primeiramente, as diferenças entre valores médios de células vizinhas são definidas como:

$$\Delta_{-} := U_i - U_{i-1} \hspace{3cm} \Delta_{+} := U_{i+1} - U_{i}$$

e
$$\Delta_{t} = \Delta_{-} + \Delta_{+}$$

introduzimos indicador de localização não dimensional $f$, definido por

$$f := \dfrac{\Delta_{-}}{\Delta_{t}} $$

que indica a localização relativa de $U_i$ em relação a $U_{i-1}$ e $U_{i+1}$. 

Observa-se que, se $U_{i-1}$, $U_i$ e $U_{i+1}$ forem monotônicos isto é, crescentes ou decrescentes, tem-se $0 \leq f \leq 1$. 

Para todas as outras condições, $U_{i}$ é um extremo local, e tem-se $−1 < f < 0$ ou $1 < f < +\infty$.


**Dedução das Equações das Retas**

**Regras TVD**

* Regra dos Extremos Locais: Se o valor da célula $U_i$ for um pico ou um vale (maior ou menor que seus dois vizinhos imediatos), a inclinação tem que ser zero. Retornamos à primeira ordem para manter a estabilidade. 

* Regra das Bordas (TVD): A inclinação não pode ser negativa se os dados vizinhos estiverem crescendo, nem positiva se estiverem diminuindo. Além disso, os valores nas extremidades (bordas) da nossa linha reconstruída não podem ultrapassar os valores médios das células vizinhas ($U_{i-1}$ e $U_{i+1}$). Se a inclinação for muito íngreme, a linha vai "invadir" o espaço vertical das células vizinhas, gerando os overshoots.

### **Alta Resolução TVD**

Como vimos, a região TVD garante que não haverá oscilações espúrias. No entanto, a região TVD pura é bastante ampla e permite esquemas que são muito "seguros", mas que dissipam (borram) demais a solução — ou seja, eles destroem a nitidez de uma onda de choque ou de um gradiente acentuado.

Para termos Alta Resolução (manter a frente de choque nítida sem oscilar), o autor estabelece uma regra gráfica fascinantemente simples: as inclinações TVD de alta resolução devem estar situadas entre as duas menores inclinações dentre as quatro retas limite ($s_{-}$, $s_{+}$, $2s_{-}$, $2s_{+}$)



#### **Limite Inferior** 

Se queremos a maior segurança possível dentro da zona de alta resolução, nós sempre escolhemos a menor inclinação possível que ainda nos dá segunda ordem.Matematicamente, nós olhamos para a inclinação fornecida pela célula à esquerda ($2f$) e pela célula à direita ($2(1-f)$) e pegamos a menor das duas. É literalmente aplicar a função "mínimo módulo" (minimum modulus).

A função do limitador **minmod** é dada por:
$$\phi(f) = \min[2f, 2(1-f)]$$

Ele forma o limite inferior exato da região TVD de alta resolução.  Na prática de CFD, o minmod é muito robusto, mas ele é o mais difusivo dos limitadores de alta resolução. Ele tende a "arredondar" os cantos de ondas quadradas mais do que gostaríamos.

#### **Limite Superior** 

O limitador superbeeSe o minmod é conservador, o limitador superbee (criado por P.L. Roe) é agressivo. Ele tenta manter a inclinação o mais íngreme possível sem violar a regra TVD.
* Ele forma o limite superior da região TVD de alta resolução.  
* Sua equação acompanha o topo da região sombreada no gráfico, trocando de comportamento dependendo da região do parâmetro $f$.  
* O superbee é excelente para resolver descontinuidades bruscas, mas pode transformar curvas suaves em perfis que parecem "degraus" (um efeito chamado de steeping).

#### **Limitadores Suaves**

Entre a dissipação extrema do minmod e a compressão agressiva do superbee, a maioria dos esquemas comerciais e acadêmicos prefere limitadores intermediários que balanceiam essas características. Muitos deles são simétricos, o que significa que se comportam da mesma forma independentemente da direção do fluxo, satisfazendo a condição $\phi(1-f) = \phi(f)$.

**van Leer:** Um limitador suave muito clássico dado por 
$$\phi(f) = 4f(1-f)$$

**Barth-Jespersen (ou MC - Monotonized Central):** Muito utilizado em malhas não estruturadas (perfeito para Galerkin Descontínuo). É dado por 
$$\phi(f) = \min[1, 4f, 4(1-f)]$$

#### Exemplo Dummy

Vamos montar um cenário clássico e muito simples: uma onda de choque em um escoamento 1D. Imagine que a variável $U$ seja a densidade do fluido.

Temos um choque perfeitamente vertical (um degrau) posicionado na nossa malha. Os valores médios das células vizinhas são:
* Célula $i-1$: $U_{i-1} = 1$ (alta densidade)
* Célula $i$: $U_i = 0$ (baixa densidade)
* Célula $i+1$: $U_{i+1} = 0$ (baixa densidade)

O nosso choque está exatamente entre a célula $i-1$ e a célula $i$. Vamos ver o que acontece quando tentamos fazer a reconstrução linear dentro da célula $i$ (onde $U_i = 0$) para calcular o fluxo na face direita.

**O Problema: Alta Ordem SEM Limitador**

Se formos usar um esquema de segunda ordem "ingênuo" (como uma diferença central clássica ou o esquema de Fromm), a inclinação $s_i$ da reta dentro da célula $i$ é calculada pegando a média entre os vizinhos:

$$s_i = \frac{U_{i+1} - U_{i-1}}{2\Delta x}$$

Substituindo os nossos valores numéricos:

$$s_i = \frac{0 - 1}{2\Delta x} = -\frac{1}{2\Delta x}$$

Agora, o algoritmo de volumes finitos ou Galerkin Descontínuo precisa saber qual é o valor da densidade bem na fronteira da célula, na interface direita ($x_{i+1/2}$), para calcular o fluxo que vai para a próxima célula.

A equação da reta (que você mesmo deduziu) extrapolada para a borda direita é:

$$U(x_{i+1/2}) = U_i + s_i \frac{\Delta x}{2}$$

Substituindo o $U_i = 0$ e o $s_i$ que acabamos de calcular:

$$U(x_{i+1/2}) = 0 + \left( -\frac{1}{2\Delta x} \right) \frac{\Delta x}{2}$$

$$U(x_{i+1/2}) = -0.25$$

Aqui o código "quebra" a física! Nós acabamos de calcular que, bem na interface da célula, a densidade do fluido é -0.25. Criamos massa negativa. No próximo passo do Runge-Kutta, o solver vai calcular a velocidade do som ($c = \sqrt{\gamma P / \rho}$), vai tentar tirar a raiz quadrada de um número negativo, vai cuspir um NaN (Not a Number) e sua simulação vai travar. Esse undershoot de -0.25 é a oscilação espúria na prática.

**A Solução: Alta Ordem COM Limitador (Minmod)**

Agora vamos repetir a exata mesma conta na célula $i$, mas ativando o limitador de inclinação minmod.

Primeiro, o limitador obriga você a olhar para as inclinações separadamente (os nossos famosos $s_{-}$ e $s_{+}$ da dedução anterior):Inclinação à esquerda:

$$s_{-} = \frac{U_i - U_{i-1}}{\Delta x} = \frac{0 - 1}{\Delta x} = -\frac{1}{\Delta x}$$

Inclinação à direita:

$$s_{+} = \frac{U_{i+1} - U_i}{\Delta x} = \frac{0 - 0}{\Delta x} = 0$$

A regra do limitador minmod é estrita: se as inclinações vizinhas tiverem sinais opostos (o que indica um pico ou vale) ou se alguma delas for zero (o que indica um patamar), a inclinação reconstruída deve ser zero para evitar criar novos extremos locais.

Como $s_{+}$ é $0$, o minmod atua como um sensor e "corta" a inclinação:

$$s_i = \text{minmod}(s_{-}, s_{+}) = 0$$

Vamos agora recalcular o valor na interface direita da célula $i$:

$$U(x_{i+1/2}) = U_i + s_i \frac{\Delta x}{2}$$

$$U(x_{i+1/2}) = 0 + 0 \cdot \frac{\Delta x}{2} = 0$$

Física preservada! O limitador percebeu que tentar manter a segunda ordem ali ia empurrar a densidade para baixo de zero. Ele forçou a inclinação para zero (caindo localmente para primeira ordem) apenas naquela célula. A densidade na borda continuou 0, nenhum valor negativo foi passado para o cálculo de fluxo, e a simulação segue estável sem crashar.

## **Slope Limiters no DG**

### **Minmod no contexto do DG**

Na teoria que vimos usando o Diagrama de Sweby e o parâmetro $f$, nós normalizamos tudo para ficar mais fácil de desenhar gráficos adimensionais. Mas na hora de programar, calcular divisões como $f = \frac{\Delta_-}{\Delta_+}$ é perigoso, porque $\Delta_+$ pode ser zero (divisão por zero quebra o código). Por isso, os solvers implementam o Minmod operando diretamente sobre as diferenças ou inclinações brutas.

A definição matemática clássica da função minmod para um conjunto de valores $(a_1, a_2, \dots, a_n)$ é:
* Se todos os valores tiverem o mesmo sinal, ela retorna o valor com o menor módulo, mantendo o sinal.
* Se houver qualquer troca de sinal entre os valores, ela retorna zero.

matematicamente (Warburton, 2008)

$$
m(a_1, \dots, a_m) = 
\begin{cases}
s \min_{1 \leq i \leq m} |a_i|, & \quad |s| = 1 \\
0, & \quad \text{caso contrario}
\end{cases}
$$

em que

$$s = \dfrac{1}{m} \sum_{i=1}^m \text{sign}(a_i)$$

---

#### **Passo a passo para codar o** `minmod`

Considere que a matriz v contém as diferenças que queremos comparar (por exemplo, a primeira linha é $\Delta_-$ de todas as células, e a segunda linha é $\Delta_+$).

```
m = np.size(v,0); 
mfunc = np.zeros((np.size(v,1)))
```

* m é a quantidade de itens que estamos comparando por célula (geralmente 3, comparando a esquerda, o centro e a direita).
* mfunc é o vetor de saída, já inicializado com zeros. A regra de "retornar zero se os sinais forem diferentes" já está pré-aplicada para todo o domínio. Só precisamos alterar os locais onde os sinais são iguais.

```s_1 = sum(np.sign(v),0)/m```

* A função `np.sign(v)` transforma todos os números positivos em 1, negativos em -1 e zeros em 0.
* Ao somar esses sinais (sum(..., 0)) e dividir pela quantidade de itens (m), o resultado s_1 só tem três possibilidades de valores absolutos para cada célula:
  * Se todos forem positivos, a soma é $m$, então $s_1 = 1$.
  * Se todos forem negativos, a soma é $-m$, então $s_1 = -1$.
  * Se houver sinais misturados (ex: um positivo e um negativo), a soma se cancela e $\vert{}s_1\vert{} < 1$.

```ids = (np.flatnonzero(abs(s_1)==1)).astype(int)```

* O código filtra (np.flatnonzero) e guarda os índices (ids) apenas das células onde o módulo da soma dos sinais deu exatamente 1.
* Ou seja, ele acabou de identificar perfeitamente as regiões monótonas da simulação. Onde isso for falso, é um pico ou vale local (sinais opostos), e o valor final continuará sendo o zero da inicialização.

```
if(len(ids) != 0):
    mfunc[ids] = s_1[ids] * np.amin(abs(v[:,ids]),axis=0)
```

* Apenas para os índices válidos (ids), o código pega o menor valor absoluto entre os candidatos (np.amin(abs(v...))) e multiplica pelo sinal correto (s_1[ids]), garantindo que a inclinação devolvida não crie novos extremos locais.

Essa implementação faz a exata mesma coisa que vimos antes:
* Evitar Picos Falsos: Se a inclinação da esquerda aponta para cima ($+$) e a da direita aponta para baixo ($-$), o código cai no vetor de zeros. A inclinação reconstruída vira zero. Retornamos para primeira ordem, cortando a oscilação na raiz, assim como a "Regra 1" do texto do Zou.
* Não extrapolar os vizinhos: Ao usar o `np.amin`, ele sempre escolhe o delta mais restritivo, garantindo que a reta reconstruída fique dentro daquelas zonas de segurança.

---

In [ ]:
import numpy as np
def minmod(v):   
# function mfunc = minmod(v)
# Purpose: Implement the midmod function v is a vector
    m = np.size(v,0); 
    mfunc = np.zeros((np.size(v,1)))
    s_1 = sum(np.sign(v),0)/m    
    ids = (np.flatnonzero(abs(s_1)==1)).astype(int)   
    if(len(ids) != 0):
        mfunc[ids] = s_1[ids] * np.amin(abs(v[:,ids]),axis=0)                   
    return mfunc

#### **Exemplo passo-a-passo**

Vamos criar a nossa matriz v de tamanho $2 \times 7$, ou seja, vamos aplicar o limitador em 7 células simultaneamente com as inclinações somente a esquerda e a direita. Vamos escolher valores de inclinação que representam todas as situações físicas possíveis que o fluido pode enfrentar:

$$
v = \begin{bmatrix} 
2 & 6 & -2 & -5 & 3 & -4 & 0 \\ 
5 & 3 & -4 & -1 & -2 & 2 & 4 
\end{bmatrix} 
\begin{array}{l} 
\leftarrow \text{Linha 1: Inclinações à Esquerda } (s_-) \\ 
\leftarrow \text{Linha 2: Inclinações à Direita } (s_+) 
\end{array}
$$

**O que cada uma dessas 7 células representa fisicamente?**  
* Célula 1 ($2$ e $5$): Fluido subindo suavemente, depois sobe rápido. (Monótono crescente)
* Célula 2 ($6$ e $3$): Fluido subindo rápido, depois sobe suavemente. (Monótono crescente)
* Célula 3 ($-2$ e $-4$): Fluido descendo suavemente, depois desce rápido. (Monótono decrescente)
* Célula 4 ($-5$ e $-1$): Fluido descendo rápido, depois desce suavemente. (Monótono decrescente)
* Célula 5 ($3$ e $-2$): Fluido sobe, depois desce. (Pico local / Overshoot)
* Célula 6 ($-4$ e $2$): Fluido desce, depois sobe. (Vale local / Undershoot)
* Célula 7 ($0$ e $4$): Fluido vem reto (patamar), depois sobe. (Início de uma rampa)

**Passo 1: Inicialização**

O código detecta que m = 2 (duas linhas para comparar) e cria nosso vetor de resposta cheio de zeros para as 7 células:

`mfunc = [0, 0, 0, 0, 0, 0, 0]`

**Passo 2: Pegando os Sinais** `(np.sign(v))`

O NumPy transforma tudo em $1$, $-1$ ou $0$:

$$\text{sign}(v) = \begin{bmatrix} 1 & 1 & -1 & -1 & 1 & -1 & 0 \\ 1 & 1 & -1 & -1 & -1 & 1 & 1 \end{bmatrix}$$

**Passo 3: Calculando a soma média dos sinais** `(s_1)`

O código soma as colunas e divide por m (que é 2):
* Célula 1: $(1 + 1) / 2 = 1$
* Célula 2: $(1 + 1) / 2 = 1$
* Célula 3: $(-1 - 1) / 2 = -1$
* Célula 4: $(-1 - 1) / 2 = -1$
* Célula 5: $(1 - 1) / 2 = 0$
* Célula 6: $(-1 + 1) / 2 = 0$
* Célula 7: $(0 + 1) / 2 = 0.5$

Vetor resultante: `s_1 = [1, 1, -1, -1, 0, 0, 0.5]`

**Passo 4: Filtrando onde os sinais são iguais (ids)**

A linha abs(s_1) == 1 pergunta: Onde o valor absoluto deu exatamente 1? Isso só acontece onde os sinais de $s_-$ e $s_+$ concordam perfeitamente.
* `ids = [0, 1, 2, 3]` (que correspondem às nossas Células 1, 2, 3 e 4).
* As Células 5, 6 e 7 foram descartadas. O `mfunc` delas vai continuar sendo o zero da inicialização! O limitador já matou a oscilação nas raízes.

**Passo 5: Pegando o mínimo e multiplicando pelo sinal**

Agora o código visita as células válidas, pega o menor valor em módulo (comando np.amin(abs(v))) e multiplica pelo sinal correto (s_1):
* Célula 1: Menor entre $\vert{}2\vert{}$ e $\vert{}5\vert{}$ é $2$. Multiplica pelo sinal $1 \rightarrow \mathbf{2}$
* Célula 2: Menor entre $\vert{}6\vert{}$ e $\vert{}3\vert{}$ é $3$. Multiplica pelo sinal $1 \rightarrow \mathbf{3}$
* Célula 3: Menor entre $\vert{}-2\vert{}$ e $\vert{}-4\vert{}$ é $2$. Multiplica pelo sinal $-1 \rightarrow \mathbf{-2}$
* Célula 4: Menor entre $\vert{}-5\vert{}$ e $\vert{}-1\vert{}$ é $1$. Multiplica pelo sinal $-1 \rightarrow \mathbf{-1}$

**Resultado Final**

O vetor devolvido pela função minmod para as nossas 7 células será:

$$\text{mfunc} = [2, 3, -2, -1, 0, 0, 0]$$

Veja que elegância: as quatro primeiras células mantiveram a menor inclinação possível (para garantir estabilidade na parte lisa da onda), e as três últimas células foram "chapadas" em zero (primeira ordem) porque o limitador detectou que ali havia perigo de criar uma oscilação irreal ou lidar com um degrau abrupto!


In [ ]:
## Usando nossa funcao

vv = np.array([[2 , 6 , -2 , -5 , 3 , -4 , 0],
               [5 , 3 , -4 , -1 , -2 , 2 , 4]])

mfu = minmod(vv)
mfu

array([ 2.,  3., -2., -1.,  0.,  0.,  0.])

### **Slope Limiter de ordem N**

De maneira geral, o que fizemos até aqui foi desenvolver uma estratégia capaz de dizer qual elemento precisa ser ''limitado''. 

Se a nossa solução é representada de forma linear por partes, ela pode ser escrita como

$$u^k(x) = \bar{u}^k + (x - x_0^k) (u^k)_x$$

em que $\bar{u}^k$ é a média da célula e $x_0^k$ representa a coordenada central do elemento. Se aplicarmos a ideia de slope limiter com o uso do `minmod` $(m)$ a solução toma a forma:

$$
\Pi^1 u^k(x) = \bar{u}^k + (x - x_0^k) m \Big ((u^k)_x, \dfrac{\bar{u}^{k+1} - \bar{u}^k}{h/2}, \dfrac{\bar{u}^k - \bar{u}^{k-1}}{h/2} \Big)
$$

porém algumas modificações nos valores que serão avaliados pelo `minmod` podem ser consideradas. Uma das estratégias mais famosas é o clássico MUSCL (*Monotone Upstream-centered Scheme for Conservation Laws*) 

$$
\Pi^1 u^k(x) = \bar{u}^k + (x - x_0^k) m \left((u^k)_x, \dfrac{\bar{u}^{k+1} - \bar{u}^k}{h}, \dfrac{\bar{u}^k - \bar{u}^{k-1}}{h} \right)
$$

Apesar da intuição geométrica ser clara para uma reta (Grau $P=1$), o nosso Método DG foi construído para utilizar polinômios de graus elevados (Alta Ordem). Se aplicarmos um limitador de inclinação de forma cega em todo o domínio, nós acabaremos limitando polinômios de grau 3 ou 4 para grau 1, o que destruiria a precisão geométrica do nosso método em regiões onde o fluido é perfeitamente suave.

No entanto, existem algumas medidas que podem ser adotadas para evitar isso. Se assumirmos que a solução calculada é um polinômio de ordem N definido por partes, é natural aplicar a limitação apenas nos elementos onde forem detectadas oscilações. Para preservar a Alta Ordem, a estratégia clássica introduzida por Cockburn e Shu no método RKDG adota um passo intermediário: o **Indicador de Células Problemáticas** (*Troubled-Cell Indicator*):

1. Calcular os valores limitados das bordas, denominados $v_l^k$ e $v_r^k$ usando 
$$
\begin{align*}
v_l^k = \bar{u}^k - m(\bar{u}^k - u_l^k, \bar{u}^k - \bar{u}^{k-1},\bar{u}^{k+1} -\bar{u}^k)\\
v_r^k = \bar{u}^k - m(u_r^k - \bar{u}^k, \bar{u}^k - \bar{u}^{k-1},\bar{u}^{k+1} -\bar{u}^k)
\end{align*}
$$

2. Se $v_l^k = u^k(x_l^k)$ e $v_r^k = u^k(x_r^k)$, então não é necessário limitação e a solução local não é alterada
3. Se a limitação for necessária, calculá-se a versão limitada de $u^k$ com a estratégia MUSCL, isto é $\Pi^1 \tilde{u}^k(x)$, em que $\tilde{u}^k(x)$ é a aproximação linear de $u^k$, que é:
$$\tilde{u}^k(x) = \bar{u}^k + (x - x_0^k) (u^k)_x$$

a esse procedimento de limitador, nos referimos como um ***slope limiter* generalizado** $\Pi^N$.

> Dessa forma, para gerar o *slope limiter* generalizado, precisamos primeiro geral o linear com a estratégia MUSCL. Os códigos de ambos são apresentados na sequência.

#### **Codando o** $\Pi^1$

$$
\Pi^1 u^k(x) = \bar{u}^k + (x - x_0^k) m \left((u^k)_x, \dfrac{\bar{u}^{k+1} - \bar{u}^k}{h}, \dfrac{\bar{u}^k - \bar{u}^{k-1}}{h} \right)
$$

In [ ]:
def SlopeLimitLin(Nldof,Dhj,uhl,xl,vm1,v0,vp1):
    ulimit = uhl
    h = xl[-1,:] - xl[0,:]
    x0 = np.ones((Nldof,1)) * (xl[0,:] + h/2)
    hN = np.ones((Nldof,1))*h
    
    # Limit function
    ux = (2/hN)*np.dot(Dhj,uhl)
    slope = np.zeros((3,len(vp1)))
    slope[0,:] = ux[0,:]
    slope[1,:] = (vp1-v0)/h
    slope[2,:] = (v0-vm1)/h  
    ulimit = np.ones((Nldof,1))*v0 + (xl-x0)*(np.ones((Nldof,1))*minmod(slope))

    return ulimit

#### **Codando o** $\Pi^N$

In [ ]:
def SlopeLimitN(u,Nldof,K, xc, invV, psi,Flkp1,Frk):    
    uhatavg = u.copy()
    uhatlin = u.copy()       
    #Recovers physical solution at quadrature points from modal coefficients uhat
    uh = np.dot(psi,u)
    # Compute cell averages
    uhatavg[1:Nldof,:] = 0.
    uhavg = np.dot(psi,uhatavg)
    v = uhavg[0,:]

    # Apply slope limiter as needed.
    ulimit = uh
    eps0 = 1.0e-8

    # find end values of each element (GL quadrature points)
    ue1 = np.dot(Flkp1[0,:],u)
    ue2 = np.dot(Frk[0,:],u)

    # find end values of each element (GLL quadrature points)
    #ue1 = uh[0,:]
    #ue2 = uh[-1,:]
    # find cell averages
    vk = v
    vkm1 = np.zeros(K)
    vkm1[0] = v[0]
    vkm1[1:-1] = v[0:-2]
    vkp1 = np.zeros(K)
    vkp1[0:-2] = v[1:-1]
    vkp1[-1] = v[-1]

    # Apply reconstruction to find elements in need of limiting
    ve1 = vk - minmod(np.array([vk-ue1,vk-vkm1,vkp1-vk]))
    ve2 = vk + minmod(np.array([ue2-vk,vk-vkm1,vkp1-vk]))
    ids1 = np.nonzero(abs(ve1-ue1)>eps0)
    ids2 = np.nonzero(abs(ve2-ue2)>eps0) 
    ids = np.union1d(ids1[0],ids2[0])

    # Check to see if any elements require limiting
    condids = np.prod(np.shape(np.matrix(ids)))
    if not (condids==0):

        # create piecewise linear solution for limiting on specified elements 
        uhatlin[2:Nldof,:] = 0.
        uhl = np.dot(psi,uhatlin)

        # apply slope limiter to selected elements
        ulimit[:,ids] = SlopeLimitLin(uhl[:,ids],xc[:,ids],vkm1[ids],vk[ids],vkp1[ids])
        #print('SlopeLimitLin activated in time = ',time)
    
    #Recover modal coefficients uhat from physical solution ulimit
    uhatlim = np.dot(invV,ulimit)
    return uhatlim

### **Slope Limiter Hierárquico**

Outra abordagem que foi derivada para um *slope limiter* que não comprometa a precisão de alta ordem nas proximidades de pontos de extremo foi realizada em (Biswas, Devine, Flaherty; 1994) e expandida no trabalho Krivodonova (2009). Essa abordagem utiliza uma **limitação hierárquica** da inclinação dos coeficientes modais associados aos Polinômios de Legendre da seguinte forma:

1. Calcula os valores limitados das bordas $v_l^k$ e $v_r^k$
2. Se $v_l^k = u^k(x_l^k)$ e $v_r^k = u^k(x_r^k)$, então não é necessário limitação e a solução local não é alterada
3. Se a limitação for necessária, calculá-se a versão limitada de $v^k$ considerando o decremento para $n \in [0, N]$, em que $N$ é o grau do nosso polinômio aproximador, usando:
$$v^k_n = C^{-1} m \left(C u^k_n, u^{k+1}_{n-1} - u^{k}_{n-1}, u^{k}_{n-1} - u^{k-1}_{n-1}  \right)$$
com 
$$C = \sqrt{(2n+1)(2n+3)}$$
4. Repita até que ${v}_{k,n} = {u}_{k,n}$ (isto é, sem limitação nesse coeficiente) ou $n = 1$, caso em que a solução se reduz à solução média por célula.

> O **Limitador Hierárquico** atua sobre os coeficientes modais ($c_i$) de forma descendente. Ele começa inspecionando o coeficiente do modo mais alto (Grau $N$). Se este modo apresentar oscilações (verificado através de derivadas discretas dos modos vizinhos inferiores), o limitador atenua esse coeficiente via `minmod`. Se o modo alto foi limitado, ele desce para o modo $N-1$ e repete a verificação, parando quando a solução se torna monotônica.

Essa técnica preserva ao máximo a estrutura espectral do elemento, amortecendo apenas as altas frequências que estão ativamente causando o choque espúrio.

#### **Codando o Slope Limiter Hierárquico**

In [ ]:
def HierarchicalLimit(u,K,N,Nldof,psi,Frk,Flkp1):
# function uhatlimit = HierarchicalLimit(u);
# Purpose: Apply hierarchical slopelimiter
# to u assuming u an N’th order polynomial
    uhatavg = u.copy() 
    #Recovers physical solution at quadrature points from modal coefficients uhat
    #uh = np.dot(psi,u)
    
    # Compute cell averages
    uhatavg[1:Nldof,:] = 0.
    uhavg = np.dot(psi,uhatavg)
    v = uhavg[0,:]

    # Apply slope limiter as needed.
    ulimit = u
    eps0 = 1.0e-8
    eps1 = 1.0e-15

    # find end values of each element (GL quadrature points)
    ue1 = np.dot(Flkp1[0,:],u)
    ue2 = np.dot(Frk[0,:],u)

    # find end values of each element (GLL quadrature points)
    #ue1 = uh[0,:]
    #ue2 = uh[-1,:]

    # find cell averages
    vk = v
    vkm1 = np.zeros(K)
    vkm1[0] = v[0]
    vkm1[1:-1] = v[0:-2]   
    vkp1 = np.zeros(K)
    vkp1[0:-2] = v[1:-1]
    vkp1[-1] = v[-1]

    # Apply reconstruction to find elements in need of limiting
    ve1 = vk - minmod(np.array([vk-ue1,vk-vkm1,vkp1-vk]))
    ve2 = vk + minmod(np.array([ue2-vk,vk-vkm1,vkp1-vk]))
    ids1 = np.nonzero(abs(ve1-ue1)>eps0)
    ids2 = np.nonzero(abs(ve2-ue2)>eps0) 
    ids = np.union1d(ids1[0],ids2[0])

    # Check to see if any elements require limiting
    condids = np.prod(np.shape(np.matrix(ids)))
    if not (condids==0):
        idr = ids.copy()
        # recover modal coefficients for limiting on specified elements        
        uk = u.copy()        
        vk = u.copy()
        ukm1 = np.zeros((Nldof,K))
        ukm1[:,0] = uk[:,0]
        ukm1[:,1:-1] = uk[:,0:-2]
    
        ukp1 = np.zeros((Nldof,K))
        ukp1[:,0:-2] = uk[:,1:-1]
        ukp1[:,-1] = uk[:,-1]

        # apply hierarchical slope limiter to selected elements
        #for i in range(0,condids):
        for n in range(N,0,-1):
            C = (np.sqrt((2*n+1)*(2*n+3)))
            vk[n,idr] = C**(-1)*minmod(np.array([C*uk[n,idr],ukp1[n-1,idr]-uk[n-1,idr],uk[n-1,idr]-ukm1[n-1,idr]]))
            ulimit[n,idr] = vk[n,idr]
            idslim = np.where(abs(vk[n,idr]-uk[n,idr])<eps1)
            idr = np.delete(idr,idslim,0)
    return ulimit

### **Estratégia RKDG**

No método RKDG de Cockburn-Shu, nós deixamos os coeficientes do polinômio evoluírem no tempo soltos pela equação diferencial. Mas, a cada sub-passo do **integrador** (Euler, RK4, RKSSP), nós fazemos uma "limpeza espacial" com o limitador para garantir que a aproximação de Galerkin não perdeu a estabilidade (TVD).

Seja um trecho da implementação do nosso integrador numérico:

```
k1 = Lh(U,t)
U1 = U + (dt/6.)*k1
```

* A primeira linha é onde a física espacial acontece. O código está calculando o resíduo (a divergência dos fluxos nas bordas mais as fontes).
* A segunda linha o tempo andou. Esse é o sub-passo temporal. O problema é: como aplicamos alta ordem, esse novo estado intermediário `U1` pode ter acabado de **desenvolver uma oscilação num choque**! Se deixarmos essa oscilação viva, **ela vai entrar no Estágio 2 do Runge-Kutta**, ser amplificada, e a simulação quebra.

**A guilhotina!**

```
k1 = Lh(U,t)
U1 = U + (dt/6.)*k1
U1 = apply_limiter(U1)
k2 = Lh_operator(U1, t + 0.5 * dt)
```

Imediatamente após atualizar a variável no tempo, o código chama o limitador de inclinação. Ele olha para os coeficientes do polinômio de Legendre (ou as médias e inclinações no elemento) desse estado intermediário phi1 e, se achar um overshoot, ele "tosa" a oscilação ali mesmo.